In [17]:
import os
import json
import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
import yaml
import torch
import shutil
import time
from IPython.display import clear_output
import sys
import subprocess
import pandas as pd

In [2]:
CLASS_MAP = {i: i - 1 for i in range(1, 9)}

def decode_instances_from_json(data, H, W, box_key='bbox'):
    masks = np.array(data['masks'], dtype=np.uint8)
    class_ids = data['class_ids']
    boxes = data[box_key]

    instances = []

    for i, cls in enumerate(class_ids):
        if cls not in CLASS_MAP:
            continue

        y1, x1, y2, x2 = map(int, boxes[i])

        y1 = max(0, min(y1, H))
        y2 = max(0, min(y2, H))
        x1 = max(0, min(x1, W))
        x2 = max(0, min(x2, W))

        if y2 <= y1 or x2 <= x1:
            continue

        mini_mask = masks[:, :, i]
        if mini_mask.sum() == 0:
            continue

        full_mask = np.zeros((H, W), dtype=np.uint8)

        resized_mask = cv2.resize(
            mini_mask.astype(np.uint8),
            (x2 - x1, y2 - y1),
            interpolation=cv2.INTER_NEAREST
        )

        full_mask[y1:y2, x1:x2] = (resized_mask > 0).astype(np.uint8)

        instances.append((full_mask, CLASS_MAP[cls]))

    return instances

In [3]:
def mask_to_polygons(mask, min_area=10, eps_ratio=0.01):
    mask = (mask > 0).astype(np.uint8)

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    polygons = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        eps = eps_ratio * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, eps, True)

        poly = approx.reshape(-1, 2)
        if len(poly) >= 3:
            polygons.append(poly)

    return polygons

In [4]:
def convert_coco_masks_to_yolo(split, box_key='bbox'):
    img_dir = split
    out_img_dir = f'dataset/images/{split}'
    out_lbl_dir = f'dataset/labels/{split}'

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    files = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    for file in tqdm(files, desc=f'Converting {split}'):
        img_path = os.path.join(img_dir, file)
        json_path = img_path + '_coco.json'

        if not os.path.exists(json_path):
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue

        H, W = img.shape[:2]

        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        try:
            instances = decode_instances_from_json(data, H, W, box_key=box_key)
        except Exception as e:
            print(f"Ошибка {file}: {e}")
            continue

        label_lines = set()

        for full_mask, cls_id in instances:
            polygons = mask_to_polygons(full_mask)

            for poly in polygons:
                poly = poly.astype(np.float32)
                poly[:, 0] /= W
                poly[:, 1] /= H
                poly = np.clip(poly, 0, 1)

                flat = poly.reshape(-1)
                if len(flat) < 6:
                    continue

                line = f"{cls_id} " + " ".join(f"{p:.6f}" for p in flat)
                label_lines.add(line)

        shutil.copy2(img_path, os.path.join(out_img_dir, file))

        txt_name = file.replace('.jpg', '.txt')
        with open(os.path.join(out_lbl_dir, txt_name), 'w', encoding='utf-8') as f:
            f.write("\n".join(sorted(label_lines)))

In [5]:
convert_coco_masks_to_yolo('train', box_key='bbox')
convert_coco_masks_to_yolo('val', box_key='bbox')

print("Train:", len(os.listdir('dataset/images/train')))
print("Val:", len(os.listdir('dataset/images/val')))

Converting val: 100%|██████████| 127/127 [00:05<00:00, 23.02it/s]

Train: 2054
Val: 127


In [6]:
data = {
    'path': os.path.abspath('dataset'),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 8,
    'names': [
        'sign_0','sign_1','sign_2','sign_3',
        'sign_4','sign_5','sign_6','sign_7'
    ]
}

with open('roadsigns.yaml', 'w') as f:
    yaml.dump(data, f)

In [7]:
env = os.environ.copy()
env["KMP_DUPLICATE_LIB_OK"] = "TRUE"
env["OMP_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.Popen(
    [sys.executable, "-u", "train.py"],
    stdout=open("train_live.log", "w", encoding="utf-8"),
    stderr=subprocess.STDOUT,
    env=env
)

print("PID =", proc.pid)

PID = 14184


In [9]:
log_path = "train_live.log"

for _ in range(1000):
    clear_output(wait=True)
    if os.path.exists(log_path):
        with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()
        print("".join(lines[-30:]))
    else:
        print("Лог еще не создан")
    time.sleep(3)

Optimizer stripped from C:\Users\pirog\machinesCV\cv6\runs\segment\roadsigns_seg\weights\last.pt, 6.5MB
Optimizer stripped from C:\Users\pirog\machinesCV\cv6\runs\segment\roadsigns_seg\weights\best.pt, 6.5MB

Validating C:\Users\pirog\machinesCV\cv6\runs\segment\roadsigns_seg\weights\best.pt...
Ultralytics 8.4.34  Python-3.12.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO26n-seg summary (fused): 139 layers, 2,690,444 parameters, 0 gradients, 9.0 GFLOPs

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 6% ╸─────────── 1/16 1.2it/s 0.3s<12.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 12% ━─────────── 2/16 1.9it/s 0.5s<7.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 19% ━━────────── 3/16 2.4it/s

KeyboardInterrupt: 

In [71]:
def compute_metrics(pred_mask, gt_mask):
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)

    if pred.sum() == 0 and gt.sum() == 0:
        return 1.0, 1.0, 1.0, 0.0

    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()

    iou = tp / (tp + fp + fn + 1e-9)
    precision = tp / (pred.sum() + 1e-9)
    recall = tp / (gt.sum() + 1e-9)
    l2 = np.sqrt(((pred.astype(np.float32) - gt.astype(np.float32)) ** 2).mean())

    return iou, precision, recall, l2


def pred_union_mask(result, H, W):
    if result.masks is None or result.masks.xy is None:
        return np.zeros((H, W), dtype=bool)

    union = np.zeros((H, W), dtype=bool)

    for poly in result.masks.xy:
        if poly is None or len(poly) < 3:
            continue

        pts = np.round(poly).astype(np.int32)
        mask = np.zeros((H, W), dtype=np.uint8)
        cv2.fillPoly(mask, [pts], 1)
        union |= mask.astype(bool)

    return union


def summary(df):
    if df.empty:
        return {}

    return {
        "images": len(df),
        "iou_mean": float(df["iou"].mean()),
        "precision_mean": float(df["precision"].mean()),
        "recall_mean": float(df["recall"].mean()),
        "l2_mean": float(df["l2"].mean()),
        "iou_ge_0.5": float((df["iou"] >= 0.5).mean()),
        "iou_ge_0.75": float((df["iou"] >= 0.75).mean()),
        "iou_ge_0.9": float((df["iou"] >= 0.9).mean()),
    }

In [72]:
def evaluate_val(model, img_dir='val', box_key='bbox', imgsz=640, conf=0.25):
    rows = []

    for file in tqdm(sorted(os.listdir(img_dir)), desc='VAL'):
        if not file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            continue

        img_path = os.path.join(img_dir, file)
        json_path = img_path + '_coco.json'
        if not os.path.exists(json_path):
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue
        H, W = img.shape[:2]

        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        gt_instances = decode_instances_from_json(data, H, W, box_key=box_key)
        gt_union = np.zeros((H, W), dtype=bool)
        for m, _ in gt_instances:
            gt_union |= m.astype(bool)

        res = model.predict(img_path, imgsz=imgsz, conf=conf, verbose=False)[0]
        pred_union = pred_union_mask(res, H, W)

        iou, precision, recall, l2 = compute_metrics(pred_union, gt_union)

        rows.append({
            "image": file,
            "iou": iou,
            "precision": precision,
            "recall": recall,
            "l2": l2
        })

    df = pd.DataFrame(rows)
    return df, summary(df)

In [73]:
best_model = YOLO('runs/segment/roadsigns_seg/weights/best.pt')

val_df, val_sum = evaluate_val(
    best_model,
    img_dir='val',
    box_key='bbox',
    imgsz=640,
    conf=0.25
)

print(val_sum)
val_df.to_csv('val_metrics.csv', index=False)

VAL: 100%|██████████| 255/255 [00:09<00:00, 26.81it/s]

{'images': 127, 'iou_mean': 0.6514516878938826, 'precision_mean': 0.7513608848820325, 'recall_mean': 0.7772533445670143, 'l2_mean': 0.08200955048467465, 'iou_ge_0.5': 0.7322834645669292, 'iou_ge_0.75': 0.49606299212598426, 'iou_ge_0.9': 0.15748031496062992}


In [74]:
def load_coco_gt_union(coco, img_path, H, W):
    name = os.path.basename(img_path)
    stem = os.path.splitext(name)[0]

    img_info = None
    for im in coco["images"]:
        fn = os.path.basename(im["file_name"])
        if fn == name or os.path.splitext(fn)[0] == stem:
            img_info = im
            break

    if img_info is None:
        return None

    union = np.zeros((H, W), dtype=bool)

    anns = [a for a in coco["annotations"] if a["image_id"] == img_info["id"]]
    for ann in anns:
        seg = ann.get("segmentation", None)
        if seg is None:
            continue

        polys = [seg] if isinstance(seg[0], (int, float)) else seg

        for poly in polys:
            if len(poly) < 6:
                continue

            pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
            pts = np.round(pts).astype(np.int32)

            mask = np.zeros((H, W), dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 1)
            union |= mask.astype(bool)

    return union


def evaluate_folder_coco(model, images_dir, coco_json, imgsz=1024, conf=0.25):
    with open(coco_json, 'r', encoding='utf-8') as f:
        coco = json.load(f)

    rows = []

    for file in tqdm(sorted(os.listdir(images_dir)), desc='MY PHOTOS'):
        if not file.lower().endswith(('.jpg', '.jpeg')):
            continue

        img_path = os.path.join(images_dir, file)
        img = cv2.imread(img_path)
        if img is None:
            continue

        H, W = img.shape[:2]

        gt_union = load_coco_gt_union(coco, img_path, H, W)
        if gt_union is None:
            continue

        res = model.predict(img, imgsz=imgsz, conf=conf, verbose=False)[0]
        pred_union = pred_union_mask(res, H, W)

        iou, precision, recall, l2 = compute_metrics(pred_union, gt_union)

        rows.append({
            "image": file,
            "iou": iou,
            "precision": precision,
            "recall": recall,
            "l2": l2
        })

    df = pd.DataFrame(rows)
    return df, summary(df)

In [75]:
my_df, my_sum = evaluate_folder_coco(
    best_model,
    images_dir='my_photos/images',
    coco_json='my_photos/coco_annotations.json',
    imgsz=1280,
    conf=0.25
)

print(my_sum)
my_df.to_csv('my_photos_metrics.csv', index=False)

MY PHOTOS: 100%|██████████| 11/11 [00:02<00:00,  4.47it/s]

{'images': 10, 'iou_mean': 0.3902981189959671, 'precision_mean': 0.7524280204949665, 'recall_mean': 0.41148203666305544, 'l2_mean': 0.24899175763130188, 'iou_ge_0.5': 0.5, 'iou_ge_0.75': 0.1, 'iou_ge_0.9': 0.0}


In [76]:
report = pd.DataFrame([
    summary(val_df),
    summary(my_df)
], index=["VAL", "MY PHOTOS"])

display(report.round(4))

,images,iou_mean,precision_mean,recall_mean,l2_mean,iou_ge_0.5,iou_ge_0.75,iou_ge_0.9
VAL,127,0.6515,0.7514,0.7773,0.082,0.7323,0.4961,0.1575
MY PHOTOS,10,0.3903,0.7524,0.4115,0.249,0.5000,0.1000,0.0000
